# Customer Churn Prediction – Data Preprocessing

## Objective
Prepare the cleaned customer churn dataset for machine learning.

## Steps
1. Load the cleaned dataset.
2. Separate input features (X) and target (y).
3. Encode categorical variables.
4. Split the dataset into training and testing sets.
5. Scale numerical features where appropriate.
6. Prepare the data for model training.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cleaned_customer_churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
df.shape

(7043, 21)

In [3]:
df.isnull().sum().sum()

np.int64(0)

In [4]:
X = df.drop(columns=["customerID", "Churn"])
y = df["Churn"]

In [5]:
X.shape

(7043, 19)

In [6]:
y.shape

(7043,)

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [8]:
X_train.shape, X_test.shape

((5634, 19), (1409, 19))

In [9]:
y_train.value_counts(normalize=True) * 100

Churn
No     73.464679
Yes    26.535321
Name: proportion, dtype: float64

In [10]:
y_test.value_counts(normalize=True) * 100

Churn
No     73.456352
Yes    26.543648
Name: proportion, dtype: float64

In [11]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_features = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

In [12]:
print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numerical features: 3
Categorical features: 16


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [15]:
X_train_processed = preprocessor.fit_transform(X_train)

In [16]:
print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

Original training shape: (5634, 19)
Processed training shape: (5634, 46)


In [17]:
X_test_processed = preprocessor.transform(X_test)

In [18]:
print("Processed test shape:", X_test_processed.shape)

Processed test shape: (1409, 46)


In [19]:
y_train_encoded = y_train.map({"No": 0, "Yes": 1})
y_test_encoded = y_test.map({"No": 0, "Yes": 1})

In [20]:
print(y_train_encoded.value_counts())
print(y_test_encoded.value_counts())

Churn
0    4139
1    1495
Name: count, dtype: int64
Churn
0    1035
1     374
Name: count, dtype: int64


In [21]:
print("Training features:", X_train_processed.shape)
print("Testing features:", X_test_processed.shape)

print("Training target:", y_train_encoded.shape)
print("Testing target:", y_test_encoded.shape)

Training features: (5634, 46)
Testing features: (1409, 46)
Training target: (5634,)
Testing target: (1409,)


In [22]:
import joblib

joblib.dump(preprocessor, "../models/preprocessor.pkl")

joblib.dump(
    {
        "X_train": X_train_processed,
        "X_test": X_test_processed,
        "y_train": y_train_encoded,
        "y_test": y_test_encoded
    },
    "../data/processed/prepared_data.pkl"
)

['../data/processed/prepared_data.pkl']